In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

print(f"Versiune PyTorch: {torch.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Sistemul va antrena modelul pe: {device.type.upper()}")
if device.type == 'cuda':
    print(f"Model GPU detectat: {torch.cuda.get_device_name(0)}")
else:
    print("Atentie: Antrenamentul se face pe procesor (CPU) si va dura mai mult.")

IMG_SIZE = 224
BATCH_SIZE = 32
DATA_DIR = "../data/raw/plantvillage/"

Versiune PyTorch: 2.7.1+cu118
Sistemul va antrena modelul pe: CUDA
Model GPU detectat: NVIDIA GeForce RTX 3060


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Incepem antrenamentul pe: {device.type.upper()}\n")

IMG_SIZE = 224
BATCH_SIZE = 32
DATA_DIR = "../data/raw/plantvillage/"

transformare = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(DATA_DIR, transform=transformare)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
num_classes = len(dataset.classes)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

num_epochs = 5
print(f"--- START ANTRENAMENT (5 Epoci) pe {num_classes} clase ---")

for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    correct = 0
    total = 0
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
            
    epoch_acc = 100 * correct / total
    epoch_time = time.time() - start_time
    print(f"Epoca {epoch+1}/5 finalizata in {epoch_time:.1f} secunde | Acuratete antrenament: {epoch_acc:.2f}%")

os.makedirs('../models', exist_ok=True)
cale_salvare = '../models/vision_model_rtx.pth'
torch.save(model.state_dict(), cale_salvare)
print(f"\nModel salvat cu succes la: {cale_salvare}")

Începem antrenamentul pe: CUDA

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\nicol/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100.0%


--- START ANTRENAMENT (5 Epoci) pe 16 clase ---
✅ Epoca 1/5 finalizată în 289.6 secunde | Acuratețe antrenament: 48.44%
✅ Epoca 2/5 finalizată în 99.7 secunde | Acuratețe antrenament: 48.47%
✅ Epoca 3/5 finalizată în 96.5 secunde | Acuratețe antrenament: 48.95%
✅ Epoca 4/5 finalizată în 92.4 secunde | Acuratețe antrenament: 49.35%
✅ Epoca 5/5 finalizată în 91.2 secunde | Acuratețe antrenament: 48.96%

🏆 Model salvat cu succes la: ../models/vision_model_rtx.pth


In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

print("Reimprospatam adresele imaginilor din memorie...")

IMG_SIZE = 224
BATCH_SIZE = 32
DATA_DIR = "../data/raw/plantvillage/"

transformare = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset_fresh = datasets.ImageFolder(DATA_DIR, transform=transformare)
train_size = int(0.8 * len(dataset_fresh))
val_size = len(dataset_fresh) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset_fresh, [train_size, val_size])

train_loader_fresh = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

for param in model.parameters():
    param.requires_grad = True

optimizer_ft = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

epoci_finetuning = 5
print(f"\n--- START FINE-TUNING ({epoci_finetuning} Epoci) ---")
print(f"Antrenim pe {len(dataset_fresh.classes)} clase corecte. Vei observa cum acurateta explodeaza spre 90%!")

for epoch in range(epoci_finetuning):
    start_time = time.time()
    model.train()
    correct = 0
    total = 0
    
    for i, (inputs, labels) in enumerate(train_loader_fresh):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer_ft.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_ft.step()
        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
            
    epoch_acc = 100 * correct / total
    epoch_time = time.time() - start_time
    print(f"Epoca FT {epoch+1}/{epoci_finetuning} gata in {epoch_time:.1f} sec. | Acuratete noua: {epoch_acc:.2f}%")

cale_noua = '../models/vision_model_rtx_finetuned.pth'
torch.save(model.state_dict(), cale_noua)
print(f"\nModelul suprem a fost salvat la: {cale_noua}")

Reîmprospătăm adresele imaginilor din memorie...

--- START FINE-TUNING (5 Epoci) ---
Antrenăm pe 15 clase corecte. Vei observa cum acuratețea explodează spre 90%!
🔥 Epoca FT 1/5 gata în 121.1 sec. | Acuratețe nouă: 82.98%
🔥 Epoca FT 2/5 gata în 103.1 sec. | Acuratețe nouă: 97.81%
🔥 Epoca FT 3/5 gata în 108.1 sec. | Acuratețe nouă: 98.89%
🔥 Epoca FT 4/5 gata în 104.1 sec. | Acuratețe nouă: 99.35%
🔥 Epoca FT 5/5 gata în 103.0 sec. | Acuratețe nouă: 99.52%

🏆 Modelul suprem a fost salvat la: ../models/vision_model_rtx_finetuned.pth
